# Review-Based Rating Construction

Background

Customer reviews often contain evaluations of several different parts of the purchasing experience within the same text. However, the dataset provides only a single overall customer rating for each review, which does not distinguish between evaluations of the product, seller, and delivery. A single review may, for example, praise the product while criticizing the delivery or the seller's responsiveness. To construct ratings for these different dimensions, the review must therefore first be decomposed into specific evaluative aspects.

In [15]:
import sys
from pathlib import Path

# Add the project root to Python's import path so modules in src/ can be imported.
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

## Sentence Embedding for Aspect Candidate Discovery

Background

The later aspect-extraction step requires a predefined set of aspect categories. Rather than allowing the model to generate arbitrary aspect names, we first identify a manageable set of candidate aspects from the reviews themselves. This helps produce more consistent outputs and makes the extracted aspects easier to compare and aggregate later, particularly when evaluating different parts of the purchasing experience, such as the product, seller, and delivery.

To identify these candidates, we examine groups of reviews that are similar according to their sentence embeddings. Sentence embeddings represent each review as a fixed-length numerical vector, with reviews that are similar with respect to the embedding model's training objective tending to have similar representations. However, we do not assume that nearest neighbors are necessarily equivalent in meaning or that every review in a neighborhood expresses the same aspect.

Examining several related reviews together can make recurring themes easier to recognize than examining reviews in isolation. These recurring themes provide exploratory evidence that can then be used to manually select, consolidate, and define the candidate aspect categories for the subsequent extraction step.


### Methodology
This notebook samples 2,000 non-empty English reviews, converts them into sentence embeddings, and retrieves the eight nearest neighbors for each review. It provides functions for inspecting individual neighborhoods as well as a randomly selected set of neighborhoods.

The resulting groups of similar reviews are used as exploratory material for the subsequent manual identification and definition of candidate aspects. Random seeds are specified for the sampling procedures to make the inspection process reproducible.


In [2]:
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModel
import torch

c:\Users\19148\anaconda3\envs\pydata-book\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


### 1. Load the review data and construct a sample

This section loads the translated review dataset and constructs a fixed sample of 2,000 reviews containing non-empty English review comments. Reviews with missing or empty comments are removed before sampling, since they cannot provide useful information for sentence embedding or qualitative inspection.

Rather than embedding the entire table, we work with a fixed sample of 2,000 reviews. This keeps the exploratory analysis computationally manageable while still providing a sufficiently broad collection of reviews for inspecting recurring themes and potential aspect categories.

In [3]:
reviews = pd.read_csv(r"../src/reviews_translated.csv", index_col='order_id', 
                        parse_dates=['review_creation_date', 'review_answer_timestamp'], 
                        date_format="%Y-%m-%d %H:%M:%S")

sample = (
    reviews[
        reviews["review_comment_en"].notna() &
        reviews["review_comment_en"].str.strip().ne("")
    ]
    .sample(n=2000, random_state=42)
    .reset_index(drop=True)
)
texts = sample["review_comment_en"].tolist()

### 2. Load the embedding model

The all-MiniLM-L6-v2 model is used to transform each review into a dense vector representation that captures aspects of its semantic content. The model is loaded from the Hugging Face Hub using AutoTokenizer and AutoModel.

Because this notebook only uses the pretrained model to generate embeddings and does not perform any training or parameter updates, the model is placed in evaluation mode (`model.eval`), different from the training mode.

The computations are performed on a GPU when CUDA is available and otherwise fall back to the CPU. Moving the model to the selected device allows the subsequent embedding computation to use the available hardware.

In [4]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Device:", device)

Device: cpu


### 3. Encode reviews as sentence embeddings

Each review is converted into a fixed-dimensional sentence embedding using the pretrained model loaded above. The resulting embedding represents each review as a numerical vector in a shared vector space. Reviews with similar semantic content are expected to have more similar embeddings, although embedding similarity does not guarantee that two reviews express exactly the same meaning or aspect.

Batch processing is used to reduce computational overhead, while a maximum sequence length of 128 tokens provides a practical limit for this exploratory analysis.Reviews longer than this limit are truncated.

`encode_sentences` handles the tokenization, model inference, pooling, and batching required to convert the review texts into embedding vectors.

In [5]:
from src.embeddings import encode_sentences

embeddings = encode_sentences(
    texts,
    tokenizer=tokenizer,
    model=model,
    device=device,
    batch_size=32,
    max_length=128
)

print(embeddings.shape)

torch.Size([2000, 384])


### 4. Compute semantic similarity and identify nearest neighbors

The pairwise similarity matrix is used to identify semantically similar reviews. Because the embeddings produced by the encoding function are normalized, their dot product is equivalent to cosine similarity. For each review, the eight reviews with the highest similarity scores are treated as its nearest neighbors.

A review is its own most similar item, so the diagonal of the similarity matrix is explicitly excluded before nearest-neighbor retrieval. Without this step, every review would trivially appear as its own first neighbor.

In [6]:
similarity = embeddings @ embeddings.T

k = 8

similarity_for_knn = similarity.clone()

# Exclude each review itself
similarity_for_knn.fill_diagonal_(-float("inf"))

# torch.topk retrieves k highest similarity scores and their review indices
neighbor_similarity, neighbor_indices = torch.topk(
    similarity_for_knn,
    k=k,
    dim=1
)

### 5. Inspect an individual review and its nearest neighbors

A single neighborhood is inspected first to verify that the nearest-neighbor retrieval behaves as expected. This provides a direct qualitative check of whether high similarity scores correspond to reviews with genuinely related content rather than merely sharing superficial vocabulary.

In [7]:
i = 2

print("Original review:")
print(sample.loc[i, "review_comment_en"])

print("\nNearest neighbors:")

for rank in range(k):

    j = neighbor_indices[i, rank].item()
    score = neighbor_similarity[i, rank].item()

    print(f"\n{rank + 1}. similarity = {score:.3f}")
    print(sample.loc[j, "review_comment_en"])

Original review:
The necklace is beautiful, with good finish on the pendants, the string of the necklace is good, the only part of the scapular that is not very well made are the rings to hang the pendants, came a bit crooked and open.

Nearest neighbors:

1. similarity = 0.642
I love the pendants, they're beautiful, well finished, of good quality.

2. similarity = 0.589
The shop itself is good but I asked for a necklace and came another completely different

3. similarity = 0.525
I got the defective pendant, broken.

4. similarity = 0.510
the pendants came peeling, one of them crushed and with the gold finish with defects.

5. similarity = 0.437
The chandelier is very beautiful, well finished, delicate painting. I recommend it!

6. similarity = 0.432
The watch itself is not so beautiful. It is very large and the material seems fragile for price. But the store delivered before and comes packed.

7. similarity = 0.429
Beautiful watch, very elegant, unbelievable in front of the value of 

### 6. Encapsulate neighbor inspection in reusable functions

The `show_neighbors` function encapsulates the display logic for one review and its nearest neighbors. The number of neighbors shown is inferred directly from `neighbor_indices.shape[1]` rather than relying on the global k variable.

Separating presentation from the embedding and retrieval steps makes the notebook easier to read and allows the same inspection procedure to be reused for different reviews without duplicating code.

In [8]:
def show_neighbors(
    sample,
    neighbor_indices,
    neighbor_similarity,
    i,
    text_column="review_comment_en"
):
    """
    Display one review and its k nearest neighbors.
    """

    print("=" * 80)
    print(f"REVIEW {i}")
    print("=" * 80)

    print("\nOriginal:")
    print(sample.loc[i, text_column])

    print("\nNearest neighbors:")

    for rank in range(neighbor_indices.shape[1]):

        j = neighbor_indices[i, rank].item()
        score = neighbor_similarity[i, rank].item()

        print(f"\n{rank + 1}. similarity = {score:.3f}")
        print(sample.loc[j, text_column])

`inspect_random_neighbors` extends this procedure to multiple reviews. The sampling is performed without replacement, ensuring that the same review is not selected more than once within a single inspection.

In [9]:
def inspect_random_neighbors(
    sample,
    neighbor_indices,
    neighbor_similarity,
    n=10,
    text_column="review_comment_en",
    random_state=42
):
    rng = np.random.default_rng(random_state)

    indices = rng.choice(
        len(sample),
        size=n,
        replace=False
    )

    for i in indices:
        show_neighbors(
            sample,
            neighbor_indices,
            neighbor_similarity,
            i,
            text_column=text_column
        )

### 7. Apply the reusable KNN implementation

The get_knn function returns the k nearest neighbors and their cosine similarities for each embedding. It assumes that the embeddings are L2-normalized, so their dot products give cosine similarities, and each review is excluded from its own neighbors.

Here, we retrieve the eight nearest neighbors for all 2,000 reviews and inspect 20 randomly selected reviews. The resulting samples are then summarized manually to identify recurring topics that can be used to define candidate aspects.

In [10]:
from src.embeddings import get_knn

neighbors_8, similarities_8 = get_knn(
    embeddings,
    k=8
)

inspect_random_neighbors(
    sample,
    neighbors_8,
    similarities_8,
    n=20
)

REVIEW 1533

Original:
Satisfied with the product!

Nearest neighbors:

1. similarity = 0.845
I'm satisfied with the product.

2. similarity = 0.831
satisfied with the purchase made.

3. similarity = 0.788
Very satisfied with the purchase and the company!

4. similarity = 0.751
I'm satisfied with the purchase

5. similarity = 0.743
Satisfied!

6. similarity = 0.723
Satisfaction with the products and everything delivered in the right time. Thank you!

7. similarity = 0.718
Product received as announced. Customer satisfied.

8. similarity = 0.707
Very satisfied with the purchase and the service
REVIEW 1431

Original:
nice

Nearest neighbors:

1. similarity = 0.640
good

2. similarity = 0.640
good

3. similarity = 0.592
Very nice.

4. similarity = 0.592
Very nice.

5. similarity = 0.528
perfect

6. similarity = 0.503
Good.

7. similarity = 0.503
Good.

8. similarity = 0.503
Good.
REVIEW 400

Original:
Very nice.

Nearest neighbors:

1. similarity = 1.000
Very nice.

2. similarity = 0.731


### Appendix: Aspect Candidates

The preceding exploratory and diagnostic investigation produced the following aspect candidates, which will be used later in the aspect-extraction step with an LLM.

**aspect**: a short, specific `snake_case` label for what is being evaluated.

- "overall_satisfaction"
- "product_quality"
- "product_condition_damage"
- "product_completeness"
- "product_color_accuracy"
- "product_description_accuracy"
- "product_appearance"
- "wrong_item_shipped"
- "packaging"
- "delivery_timing"
- "delivery_non_receipt"
- "price"
- "seller_contact"
- "seller_service"
- "others"

## LLM-Based Aspect Extraction

Background

This stage uses a large language model (LLM) to perform aspect-based extraction from Brazilian Portuguese e-commerce reviews. The LLM identifies the specific aspect being evaluated, based on the candidate aspects established in the previous section, the attitude toward that aspect, and, where relevant, its discourse role and concession relation. These structured evaluations are subsequently used to construct ratings for three broader categories: Product, Seller, and Delivery.

### Methodology

The procedure begins by preparing the review data and initializing the LLM with the predefined extraction prompt and response schema. Each unique, non-empty Portuguese review is then analyzed to identify evaluative spans, their specific aspects, attitudes, and discourse characteristics, with the results saved incrementally for reproducibility and recovery. The extracted results are subsequently converted into a structured aspect-level dataset and checked for invalid discourse annotations. Multiple evaluations of the same aspect within a review are then consolidated using discourse role and relation to determine their relative contribution, with opposing evaluations allowed to offset one another. Finally, overall and unassignable evaluations are excluded from the specific-aspect rating system, the remaining aspects are grouped into Product, Seller, and Delivery, and aspect-level evaluations are aggregated into category ratings on a 1–5 scale.

In [3]:
import os
from google import genai
from google.genai import types
import pandas as pd
import time
import json
from dotenv import load_dotenv

### 1. Load resources and initialize the LLM

This section prepares the review dataset and LLM.

The JSON schema constrains the structure of the LLM response. It is provided with each request to enforce a machine-readable and consistent output format.

The prompt contains the actual instructions defining what counts as evaluative content, how aspects and attitudes should be assigned, and how discourse relations should be annotated (see [prompt](../src/aspect_extraction_prompt.md) and [its design](../src/aspect_extraction_prompt_design.md)). It is stored separately from the notebook so that the exact prompt used for the analysis can be preserved and passed directly to the LLM.

In [ ]:
with open(r"../src/aspect_schema.json", "r", encoding="utf-8") as f:
    schema = json.load(f)
with open(r"../src/aspect_extraction_prompt.md", "r", encoding="utf-8") as f:
    absa_instructions = f.read()

The translated review dataset is loaded. Duplicate review texts are removed on the review content rather than the index, because `order_id` is not a reliable unique identifier for the review table. Empty review comments are also excluded because they contain no text for the LLM to analyze.

In [18]:
reviews = pd.read_csv(r"../src/olist_reviews_translated.csv", index_col='order_id', 
                        parse_dates=['review_creation_date', 'review_answer_timestamp'], 
                        date_format="%Y-%m-%d %H:%M:%S")

unique_reviews = reviews[
    reviews["review_comment_message"].notna()
    & ~reviews["review_comment_message"].duplicated()
].copy()

The Gemini client provides access to the selected generative model. Since the extraction prompt is long and remains unchanged across reviews, it is placed in Gemini's cached context and reused across requests, reducing the need to repeatedly process the same instructions.

> **API key and Gemini API tier**
>
> This notebook uses the Gemini API. Before running the code, place your own API key in a `.env` file in the project root:
>
> ```text
> GEMINI_API_KEY=your_api_key_here
> ```
>
> The `.env` file is excluded from version control and should not be uploaded to GitHub.
>
> The aspect-extraction workflow uses a cached version of the prompt/model context. Therefore, a **paid Gemini API tier** is required to run this part of the notebook with caching enabled.

In [ ]:
load_dotenv()

if not os.getenv("GEMINI_API_KEY"):
    raise RuntimeError("GEMINI_API_KEY is not set.")

client = genai.Client()
MODEL = "gemini-3.1-flash-lite"

cache = client.caches.create(
    model=MODEL,
    config=types.CreateCachedContentConfig(
        system_instruction=absa_instructions,
        ttl="36000s",  # cache lifetime; adjust if needed
    ),
)

print("Created cache:", cache.name)

Created cache: cachedContents/rwh1ipfv44j4x0dh6fpdwcx5y12fer68si1irny6


### 2. Run and persist the aspect extraction

analyze_review() sends an individual Portuguese review to Gemini and returns the structured aspect analysis, token-usage information, and the raw response. The analysis is processed incrementally and saved immediately to a JSONL file so that the long-running computation can be resumed without repeating successfully completed reviews.

In [20]:
from src.aspect_extraction import analyze_review

The analysis is computationally expensive and may be interrupted by API limits or other errors, so results are persisted after each review rather than being kept only in memory. JSONL is suitable for this purpose because new records can be appended without rewriting the existing results.

Before processing begins, previously successful records are identified from the output file. A record with error = None is considered completed, including cases where the model returns an empty extraction ({"spans": []}), since an empty extraction is a valid model response rather than an API failure. Malformed or incomplete JSONL records are ignored because they may result from an interrupted write.

The remaining reviews are then processed one at a time. `time.perf_counter()` records the elapsed time for each review, allowing the computational performance of the extraction process to be examined later.

A specific retry mechanism handles `429 RESOURCE_EXHAUSTED` errors, which indicate that the Gemini resource or quota limit has temporarily been reached. The request is paused and retried rather than being recorded as a permanent failure. Other exceptions are recorded in the JSONL output, allowing the analysis to continue with subsequent reviews.

In [21]:
# ============================================================
# 1. Load IDs that have already been successfully recorded
# ============================================================
output_file = r"../data/processed/review_analysis.jsonl"
completed_ids = set()

if os.path.exists(output_file):
    with open(output_file, "r", encoding="utf-8") as f:
        for line in f:
            try:
                record = json.loads(line)

                # Only treat successful records as completed
                if record.get("error") is None:
                    completed_ids.add(record["id"])

            except (json.JSONDecodeError, KeyError):
                # Ignore malformed/incomplete lines
                continue

print(f"Found {len(completed_ids)} completed reviews.")


# ============================================================
# 2. Select reviews that still need to be processed
# ============================================================

remaining = unique_reviews[
    ~unique_reviews.index.isin(completed_ids)
]

print(f"Remaining reviews: {len(remaining)}")


# ============================================================
# 3. Process reviews
# ============================================================

with open(output_file, "a", encoding="utf-8") as f:

    for i, (idx, row) in enumerate(remaining.iterrows(), start=1):

        review_pt = row["review_comment_message"]

        print(
            f"Processing {i}/{len(remaining)} | ID: {idx}"
        )

        start_time = time.perf_counter()

        # ----------------------------------------------------
        # Retry loop
        # ----------------------------------------------------

        while True:

            try:
                analysis, usage, response = analyze_review(review_pt, cache.name, client, schema, model=MODEL)

                # Successful request
                elapsed = time.perf_counter() - start_time

                record = {
                    # Original dataset information
                    "id": idx,
                    "review_pt": review_pt,

                    # Gemini analysis
                    "analysis": analysis,

                    # Usage / performance information
                    "input_tokens": usage["input_tokens"],
                    "output_tokens": usage["output_tokens"],
                    "cached_tokens": usage["cached_tokens"],
                    "time_seconds": elapsed,

                    # Error information
                    "error": None,
                }

                break


            except Exception as e:

                error_text = str(e)

                if "429" in error_text and "RESOURCE_EXHAUSTED" in error_text:

                    print("\n" + "=" * 80)
                    print("GEMINI 429 ERROR")
                    print(error_text)
                    print("=" * 80 + "\n")

                    # Temporary: wait before retrying
                    time.sleep(30)

                    continue



                # ------------------------------------------------
                # Any non-429 error
                # ------------------------------------------------

                elapsed = time.perf_counter() - start_time

                print(
                    f"Error for ID {idx}: {error_text}"
                )

                record = {
                    "id": idx,
                    "review_pt": review_pt,
                    "analysis": None,
                    "input_tokens": None,
                    "output_tokens": None,
                    "cached_tokens": None,
                    "time_seconds": elapsed,
                    "error": error_text,
                }

                break


        # ========================================================
        # 4. Save result immediately
        # ========================================================

        f.write(
            json.dumps(
                record,
                ensure_ascii=False
            ) + "\n"
        )

        f.flush()


        # ========================================================
        # 5. Small delay between successful requests
        # ========================================================

        time.sleep(0.5)

Found 36101 completed reviews.
Remaining reviews: 1
Processing 1/1 | ID: 30125d2cf4cec0bf9e4840def6c4e7b1
Error for ID 30125d2cf4cec0bf9e4840def6c4e7b1: [SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1016)


### 3. Construct the Aspect-Level Dataset

The saved LLM results are transformed into a structured DataFrame for subsequent analysis.

`load_analysis_jsonl()` reads the previously saved JSONL file and converts each line back into a Python dictionary, which is then assembled into `results_df`.

Because the LLM analysis is performed on Portuguese reviews, the JSONL file stores the Portuguese text used for extraction. The English translation is restored afterward using the Portuguese review text as the lookup key rather than the identifier, since the identifier is not unique. The English version is retained primarily to facilitate manual inspection and interpretation of the extracted results.

In [22]:
from src.aspect_extraction import load_analysis_jsonl

results = load_analysis_jsonl(output_file)
results_df = pd.DataFrame(results)

review_en_map = (
    unique_reviews
    .set_index("review_comment_message")["review_comment_en"]
)

results_df["review_en"] = results_df["review_pt"].map(review_en_map)

The original LLM output is nested because each review may contain multiple extracted spans:
```
one review
    └── spans
         ├── span 1
         ├── span 2
         └── span 3
```
For analysis, this nested structure is flattened so that each extracted evaluative span becomes one row:

> review | text | aspect | attitude | discourse_role | relation

This representation makes individual aspect evaluations easier to filter, inspect, validate, and aggregate using Pandas, which produces the main aspect-level dataset used by the subsequent validation and aggregation steps.

In [ ]:
aspect_rows = []

for _, row in results_df.iterrows():

    analysis = row["analysis"]

    if analysis is None:
        continue

    for span in analysis["spans"]:
        aspect_rows.append({
            "id": row["id"],
            "review_pt": row["review_pt"],
            "review_en": row["review_en"],
            "text": span["text"],
            "aspect": span["aspect"],
            "attitude": span["attitude"],
            "discourse_role": span["discourse_role"],
            "relation": span["relation"],
        })

aspect_df = pd.DataFrame(aspect_rows)

### 4. Validate and Correct Discourse Combinations

The LLM can occasionally generate combinations of `discourse_role` and `relation` that are individually valid enum values but logically inconsistent with the intended dependency between the two fields. These cases are isolated and manually revised based on the context of the original review.

The valid combinations are:

none       → none  
nucleus    → none  
satellite  → concession_tradeoff or concession_exclusion

In [24]:
from src.aspect_extraction import validate_discourse_relation

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
validation_df = validate_discourse_relation(aspect_df)
validation_df

,id,review_pt,review_en,text,aspect,attitude,discourse_role,relation,validation_error
22685,a77696c7b42da1121205a92ddae7b5c7,"produto não atende a sua finalidade, pois não tem o conversor de hdmi para chá.","product does not meet its purpose, because it does not have the converter of hdmi for tea.",não tem o conversor de hdmi para chá,product_completeness,negative,satellite,none,"Invalid combination: discourse_role='satellite', relation='none'"
30972,8836e9da58524060994ead4403874ad3,"Apesar de ter demorado mais do que o necessário, o ´produto chegou antes do prazo. Mas poderia ser mais rápido, já que o frete é pago. Do resto estava tudo ok.","Although it took longer than necessary, the product arrived before the deadline. But it could be faster, since the freight is paid. Of the rest everything was ok.",o frete é pago,price,negative,satellite,none,"Invalid combination: discourse_role='satellite', relation='none'"
42985,8ec58bc9d39789f8d32575cc5ce0cf0c,"O perfume é bom, tem uma fixação boa, mas não há uma grande expansão do cheiro!","The perfume is good, it has a good fixation, but there is not a great expansion of the smell!",não há uma grande expansão do cheiro,product_quality,negative,nucleus,concession_tradeoff,"Invalid combination: discourse_role='nucleus', relation='concession_tradeoff'"
48735,064df82a5ad596912cc304c6c72a42bf,"Fiz a solicitação de cancelamento do pedido e não obtive resposta da empresa. Ainda por cima consta no site como produto entregue, porém não o recebi.","I made the request to cancel the order and I did not get a reply from the company. Above all, it is listed on the website as a product delivered, but I did not receive it.",não o recebi,delivery_non_receipt,negative,nucleus,concession_tradeoff,"Invalid combination: discourse_role='nucleus', relation='concession_tradeoff'"
50699,cf0c47c42ff868bcd8e038ee2945c1b1,No site o shampoo esta como 500ml e chegou um de 300ml... Isso e propagana enganosa,On the site the shampoo is like 500ml and came a 300ml... that and deceptive propaganda,no site o shampoo esta como 500ml e chegou um de 300ml,product_description_accuracy,negative,satellite,none,"Invalid combination: discourse_role='satellite', relation='none'"
64979,9dfd36609f9bcc11f7712d98466d9aa5,no site o cartucho era original e recebi um cartucho compatível.... propaganda enganosa!!!\r\n\r\n\r\n\r\n\r\n\r\n,on the site the cartridge was original and received a compatible cartridge.... misleading propaganda!!!,no site o cartucho era original,product_description_accuracy,neutral,satellite,none,"Invalid combination: discourse_role='satellite', relation='none'"
65009,cd12238095fc7765f26477b48b9cb3b2,"Achei que a embalagem de entrega poderia ter sido de melhor qualidade. Uma caixa simples que, aparentemente, pode gerar danos ao produto. Mas a princípio o meu produto chegou em perfeito estado.","I thought the delivery package might have been of better quality, a simple box that, apparently, could cause damage to the product, but at first my product came in perfect condition.",meu produto chegou em perfeito estado,product_condition_damage,positive,nucleus,concession_tradeoff,"Invalid combination: discourse_role='nucleus', relation='concession_tradeoff'"


The purpose of this step is diagnosis rather than automatic correction. Each invalid aspect-level record is inspected separately because its intended interpretation depends on the context of the original review.

In [25]:
invalid_id = validation_df["id"]
invalid_aspect_df = aspect_df[aspect_df["id"].isin(invalid_id)]
invalid_aspect_df

,id,review_pt,review_en,text,aspect,attitude,discourse_role,relation
22684,a77696c7b42da1121205a92ddae7b5c7,"produto não atende a sua finalidade, pois não tem o conversor de hdmi para chá.","product does not meet its purpose, because it does not have the converter of hdmi for tea.",produto não atende a sua finalidade,product_quality,negative,nucleus,none
22685,a77696c7b42da1121205a92ddae7b5c7,"produto não atende a sua finalidade, pois não tem o conversor de hdmi para chá.","product does not meet its purpose, because it does not have the converter of hdmi for tea.",não tem o conversor de hdmi para chá,product_completeness,negative,satellite,none
30969,8836e9da58524060994ead4403874ad3,"Apesar de ter demorado mais do que o necessário, o ´produto chegou antes do prazo. Mas poderia ser mais rápido, já que o frete é pago. Do resto estava tudo ok.","Although it took longer than necessary, the product arrived before the deadline. But it could be faster, since the freight is paid. Of the rest everything was ok.",ter demorado mais do que o necessário,delivery_timing,negative,satellite,concession_tradeoff
30970,8836e9da58524060994ead4403874ad3,"Apesar de ter demorado mais do que o necessário, o ´produto chegou antes do prazo. Mas poderia ser mais rápido, já que o frete é pago. Do resto estava tudo ok.","Although it took longer than necessary, the product arrived before the deadline. But it could be faster, since the freight is paid. Of the rest everything was ok.",o produto chegou antes do prazo,delivery_timing,positive,nucleus,none
30971,8836e9da58524060994ead4403874ad3,"Apesar de ter demorado mais do que o necessário, o ´produto chegou antes do prazo. Mas poderia ser mais rápido, já que o frete é pago. Do resto estava tudo ok.","Although it took longer than necessary, the product arrived before the deadline. But it could be faster, since the freight is paid. Of the rest everything was ok.",poderia ser mais rápido,delivery_timing,negative,nucleus,none
30972,8836e9da58524060994ead4403874ad3,"Apesar de ter demorado mais do que o necessário, o ´produto chegou antes do prazo. Mas poderia ser mais rápido, já que o frete é pago. Do resto estava tudo ok.","Although it took longer than necessary, the product arrived before the deadline. But it could be faster, since the freight is paid. Of the rest everything was ok.",o frete é pago,price,negative,satellite,none
30973,8836e9da58524060994ead4403874ad3,"Apesar de ter demorado mais do que o necessário, o ´produto chegou antes do prazo. Mas poderia ser mais rápido, já que o frete é pago. Do resto estava tudo ok.","Although it took longer than necessary, the product arrived before the deadline. But it could be faster, since the freight is paid. Of the rest everything was ok.",estava tudo ok,overall_satisfaction,positive,none,none
42983,8ec58bc9d39789f8d32575cc5ce0cf0c,"O perfume é bom, tem uma fixação boa, mas não há uma grande expansão do cheiro!","The perfume is good, it has a good fixation, but there is not a great expansion of the smell!",O perfume é bom,product_quality,positive,none,none
42984,8ec58bc9d39789f8d32575cc5ce0cf0c,"O perfume é bom, tem uma fixação boa, mas não há uma grande expansão do cheiro!","The perfume is good, it has a good fixation, but there is not a great expansion of the smell!",tem uma fixação boa,product_quality,positive,none,none
42985,8ec58bc9d39789f8d32575cc5ce0cf0c,"O perfume é bom, tem uma fixação boa, mas não há uma grande expansão do cheiro!","The perfume is good, it has a good fixation, but there is not a great expansion of the smell!",não há uma grande expansão do cheiro,product_quality,negative,nucleus,concession_tradeoff


For cases where `discourse_role == "nucleus"`, the relation can be safely corrected to `"none"` because concession relations apply to the subordinate satellite rather than the nucleus.

For cases where `discourse_role == "satellite"` and `relation == "none"`, inspection showed that these cases were associated with causal or explanatory clauses that had been extracted as evaluative spans. Although these clauses contain genuine evaluative content, retaining both the main evaluative statement and its causal explanation would cause the same issue to be represented twice when the aspect evaluations are later mapped to the broader Product, Seller, and Delivery categories. These subordinate reason spans are therefore excluded from the subsequent aggregation.

In [26]:
invalid_aspect_df.loc[invalid_aspect_df["discourse_role"] == "nucleus", "relation"] = 'none'
invalid_aspect_df = invalid_aspect_df[
                ~((invalid_aspect_df["discourse_role"] == "satellite") & 
                (invalid_aspect_df["relation"] == "none")) ]
aspect_df[aspect_df["id"].isin(invalid_id)] = invalid_aspect_df

C:\Users\19148\AppData\Local\Temp\ipykernel_22336\784987126.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  invalid_aspect_df.loc[invalid_aspect_df["discourse_role"] == "nucleus", "relation"] = 'none'


### 5. Aggregate Aspect Evaluations

Multiple evaluations of the same specific aspect within a review are consolidated into a single aspect-level evaluation. Discourse structure is converted into an aggregation weight so that independent or nucleus evaluations contribute more strongly than concessive evaluations. Contradictory positive and negative evaluations are combined using their weighted net effect.

`aggregate_aspect_df()` performs this consolidation. The discourse hierarchy is:
$$
\mathrm{none/nucleus}>\mathrm{concession\_tradeoff}>\mathrm{concession\_exclusion}
$$
with corresponding weights:
$$
2,\qquad 1,\qquad 0.
$$

For evaluations with opposing attitudes toward the same aspect, the weights are treated as signed evidence. The resulting net weight is:
$$
\mathrm{net\ weight}=\mathrm{positive\ weight}−\mathrm{negative\ weight}.
$$
Thus, equal positive and negative evidence cancels out, while a positive or negative net weight determines the final attitude. This produces at most one final evaluation for each specific aspect within a review and prevents multiple contradictory evaluations from being carried independently into the subsequent category-level rating.

In [27]:
from src.aspect_extraction import aggregate_aspect_df

aggregated_df = aggregate_aspect_df(aspect_df)

### 6. Remove Overall Satisfaction

When a review contains specific aspect evaluations, `overall_satisfaction` is removed from the aspect-rating calculation. It represents the reviewer's overall assessment and is conceptually closer to the customer's original overall rating than to any of the three substantive dimensions being constructed.

If `overall_satisfaction` is the only evaluation in a review, it is retained for possible later comparison with the original customer rating. This prevents an overall evaluation from being incorrectly incorporated into the Product, Seller, or Delivery categories.

In [28]:
from src.aspect_extraction import remove_overall_satisfaction

aggregated_df = remove_overall_satisfaction(aggregated_df)

### 7. Calculate Category Score

Specific aspect evaluations are converted into 1–5 ratings for three broader dimensions of the e-commerce experience: **Product**, **Seller**, and **Delivery**.

Each aspect first receives an aspect-level score based on its attitude and discourse weight. The current discourse weights are:
$$
w = 
\begin{cases}
0 & \mathrm{concession\_exclusion}\\
1 & \mathrm{concession\_tradeoff} \\
2 & \mathrm{none/nucleus​}
\end{cases}
$$

Using a neutral baseline of 3, the aspect score is:
$$
S_i = 3 + w_is_i
$$
where $s_i=1$ for a positive evaluation, $0$ for a neutral evaluation, and $−1$ for a negative evaluation.

calculate_category_scores() performs the final transformation from specific aspects to the three broad dimensions:

Product
Seller
Delivery


Each specific aspect first receives an aspect-level score based on its attitude and discourse weight.

The current discourse weights are:
$$
w = 
\begin{cases}
0 & \mathrm{concession\_exclusion}\\
1 & \mathrm{concession\_tradeoff} \\
2 & \mathrm{none/nucleus​}
\end{cases}
$$
Using a neutral baseline of 3, the aspect score is:
$$
S_i = 3 + w_is_i
$$
where $s_i=1$ for a positive evaluation, $0$ for a neutral evaluation, and $−1$ for a negative evaluation.


The aspect scores belonging to the same category within a review are then averaged. The mapping is based on **whose performance/responsibility this evaluation primarily concerns**, rather than simply on the wording of the aspect label.

The current mapping is:

**Product**
- "product_quality"
- "product_condition_damage"
- "product_color_accuracy"
- "product_appearance"

**Seller**
- "product_completeness"
- "product_description_accuracy"
- "wrong_item_shipped"
- "price"
- "seller_contact"
- "seller_service"

**Delivery**
- "packaging"
- "delivery_timing"
- "delivery_non_receipt"

"others" and "overall_satisfaction" are excluded because they cannot be meaningfully incorporated into these three categories. A category that is not mentioned receives no rating rather than an artificial neutral score.

In [29]:
from src.aspect_extraction import calculate_category_scores

score_df = calculate_category_scores(aggregated_df)
score_df.head()

,review_pt,review_en,category,score
0,"\r\n\r\n\r\nDe todo o pedido feito no site, este foi o que mais demorou para entregar!!!","Of all the request made on the website, this was the most time to deliver!!!",delivery,1.0
1,\r\n\r\nHouve um impedimento relatado em outro e-mail justificando o atraso da entrega do produto .\r\nNesse momento ainda não recebi a compra.\r\nAguardo informações \r\nPatrícia,There was an impediment reported in another email justifying the delay of delivery of the product. At this time I have not yet received the purchase. I await information Patrícia,delivery,1.0
2,"\r\nApesar de de ser entregue antes do previsto, prazo de entrega é muito longo.Enfim, mais por causa dos Correios mesmo que é um lixo. Tirando isso produto em perfeito estado. Recomendo!","Despite being delivered ahead of schedule, delivery time is very long. Anyway, more because of the Post Offices even it is a garbage. Apart from this product in perfect condition. I recommend!",delivery,2.0
3,"\r\nApesar de de ser entregue antes do previsto, prazo de entrega é muito longo.Enfim, mais por causa dos Correios mesmo que é um lixo. Tirando isso produto em perfeito estado. Recomendo!","Despite being delivered ahead of schedule, delivery time is very long. Anyway, more because of the Post Offices even it is a garbage. Apart from this product in perfect condition. I recommend!",product,5.0
4,\r\nComprei um produto e entregaram outro difetente do que pedi..... E mais solicitei a devolucao ou troca e ate agora nada.... Estam se fazendo de dezentendidos...\r\nE fazer pouco caso do cliente ne..,I bought a product and they delivered another defect than I asked... and the more I asked for the return or exchange and until now nothing... they're making up their minds... and not paying any attention to the client.,seller,1.0


In [48]:
score_df.groupby("category")["score"].describe()

,count,mean,std,min,25%,50%,75%,max
category,,,,,,,,
delivery,18582.0,3.609389,1.849609,1.0,1.0,5.0,5.0,5.0
product,12883.0,3.865585,1.723366,1.0,2.0,5.0,5.0,5.0
seller,14296.0,2.831462,1.933615,1.0,1.0,1.0,5.0,5.0


In [ ]:
score_df.to_csv(
    r"../data/processed/olist_categorized_score.csv",
    index=False
)

## Summary and Next Steps

**Summary**

This notebook develops the review-based inputs needed to construct ratings for the Product, Seller, and Delivery dimensions. It first uses sentence embeddings and nearest-neighbor inspection to identify recurring topics and manually define a set of candidate aspects. It then uses an LLM to extract these aspects and their associated evaluations from the reviews, providing the information needed to calculate ratings for the three dimensions.

**Next Steps**

The resulting Product, Seller, and Delivery ratings provide the basis for the next stage of the analysis. The next notebook will aggregate these ratings and examine the resulting performance across different aspects of the purchasing experience.
